In [1]:
from time import perf_counter

from qdk.qre.application import QspApplication
from qdk.qre.profiler import QREProfiler

app = QspApplication.for_cpp_code("""
#include "qep_interop.h"

// Execute 100 million loop gates, then flush the trailing batch.
int main() {
    qdk_h(0);
    const size_t start = qdk_mz_reset(0);
    if (start > 1) return 1;
    for (size_t i = start; i < start + 1000000; ++i) {
        for (size_t j = 0; j < 50; ++j) {
            qdk_h(j);
            qdk_t(j);
        }
    }
    qdk_reset(0);
    return qdk_mz(0) > 1;
}
""")
profiler = QREProfiler.counting()
started = perf_counter()
result = profiler.profile(app)
elapsed = perf_counter() - started
count = result.totals.total_operations_count
assert count == 100_000_004
print(f"{count:,} ops in {elapsed:.3f}s (including compilation, startup, and pipe transport)")
print(f"{count / elapsed / 1e6:.3f} million ops/s, {1e9 * elapsed / count:.1f} ns/op")

100,000,004 ops in 12.357s (including compilation, startup, and pipe transport)
8.093 million ops/s, 123.6 ns/op
